# Clustering Session 3 — Practical: Real Nanoindentation Data
### FDP: Machine Learning for Materials and Metallurgical Engineering

**About this dataset:** Trost, Žak, Schaffer, Walch, Zitz, Klünsner, Leitner, Exl & Cordill, *"Explainable machine learning and feature engineering applied to nanoindentation data,"* Materials & Design, 253:113897, 2025.

**2,621 real, human-labelled nanoindentation measurements** on a commercial high-speed steel (S390 Microclean™), across four microstructural variants (BV, SV, LCV, SAV). Every indent was manually classified by a human expert into one of five phase categories — giving us genuine ground truth to benchmark unsupervised clustering against, which is unusual for a real clustering problem.

**What we'll do:** apply everything from Sessions 1–2 — the K-means algorithm, choosing K, manual similarity measures, evaluating results — to this real dataset, and directly test the paper's own headline finding: does feature engineering matter more than model choice?

## Setup — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, f1_score, confusion_matrix


## Where the Engineered Features Come From

Before loading the processed dataset, it's worth seeing what a **raw indentation measurement** actually looks like. Every nanoindentation test produces a load-displacement (P-d) curve — this is the real signal that the 13 engineered features (slope, curvature, work terms, etc.) are mathematically derived from, via dimensional analysis.

Below is one real indent's full loading-unloading curve (downsampled for a clean plot):

In [ ]:
depth_nm = [-3.0, 9.25, 17.39, 27.3, 33.91, 40.62, 45.25, 53.68, 56.18, 63.92, 66.91, 69.7,
            76.72, 78.6, 83.86, 86.12, 91.75, 91.62, 96.89, 99.57, 98.3, 100.92, 98.07, 100.37,
            99.82, 100.81, 98.93, 100.64, 101.17, 97.38, 98.79, 94.71, 95.25, 92.69, 93.4, 88.66,
            90.01, 88.66, 83.59, 84.94, 80.65, 80.71, 77.81, 74.93, 68.8, 58.96, 18.98]
load_uN = [1.45, 155.3, 318.58, 481.15, 643.19, 804.98, 966.85, 1128.16, 1289.96, 1451.24, 1612.86,
           1774.06, 1935.48, 2096.76, 2258.03, 2419.44, 2580.56, 2742.03, 2903.15, 2998.12, 2998.18,
           2998.07, 2998.14, 2998.06, 2998.13, 2998.03, 2998.24, 2998.03, 2998.04, 2851.39, 2690.38,
           2529.33, 2368.27, 2207.43, 2046.23, 1885.47, 1724.38, 1563.54, 1402.78, 1241.92, 1081.11,
           920.43, 759.86, 599.33, 439.54, 279.42, 121.34]

plt.figure(figsize=(6, 4.5))
plt.plot(depth_nm, load_uN, marker='o', markersize=3, color='darkred')
plt.xlabel("Depth (nm)")
plt.ylabel("Load (µN)")
plt.title("One real indent: loading → hold at ~3 mN → unloading")
plt.tight_layout()
plt.show()


Notice the shape: load rises as the indenter pushes in, holds briefly at the target 3 mN, then falls as the indenter withdraws — but the depth doesn't fully return to zero (the material has deformed permanently). Features like `slope_0.2` (initial unloading slope), `work_elastic`/`work_plastic` (areas under different parts of this curve), and `h_ratio` all come directly from measurements on curves exactly like this one, for each of our 2,621 indents.

## Step 1 — Load the Real Dataset

In [ ]:
TRAIN_URL = "https://raw.githubusercontent.com/vijindal/fdp-ml/main/day3/Training_set.csv"
VAL_URL = "https://raw.githubusercontent.com/vijindal/fdp-ml/main/day3/Validation_set.csv"

train = pd.read_csv(TRAIN_URL)
val = pd.read_csv(VAL_URL)

print("Training set:", train.shape)
print("Validation set:", val.shape)
train.head()


## Step 2 — Explore the Data

In [ ]:
print("Phase label distribution (training set):")
print(train['Y_name'].value_counts())
print()
print("Indentation loads present:", sorted(train['mN'].unique()))


Notice the class imbalance — `matrix` dominates, while `m6c` (M6C-type carbide) is rare. This matters later when we compare weighted vs. macro F1 scores.

## Step 3 — Filter and Prepare

In [ ]:
# Match the original study's methodology: use only the 3 mN indentation series
train = train[train.mN == 3].copy()
val = val[val.mN == 3].copy()

label_map = {'matrix': 0, 'mc-matrix': 1, 'm6c-matrix': 2, 'mc': 3, 'm6c': 4}
inv_label_map = {v: k for k, v in label_map.items()}
train['Y'] = train['Y_name'].map(label_map)
val['Y'] = val['Y_name'].map(label_map)

print("Training rows (mN=3):", len(train), "| Validation rows (mN=3):", len(val))


---
## Quick Check 1

K-means needs us to choose K in advance. There are 5 true phase labels in this dataset. Should we necessarily set K=5?

**(i)  Yes – always match K to the number of true classes**
**(ii)  Not necessarily – some labels represent transition/mixed indents, not distinct phases; the elbow method and silhouette score (Session 1) should guide K, informed by domain knowledge**
**(iii)  It doesn't matter what K is set to**

*Think about it, then check the answer below.*

**Answer: (ii)** — `mc-matrix` and `m6c-matrix` represent indents that span a matrix/carbide boundary, not a fourth and fifth distinct material phase. Domain knowledge suggests 3 real underlying phases (matrix, MC carbide, M6C carbide) is a more sensible target than blindly matching K to the label count. Let's check what the data itself says next.

## Step 4 — Choosing K: Elbow Method and Silhouette Score

In [ ]:
engineered_features = ['slope_0.2', 'norm_slope_0.2', 'curvature', 'curvature_linear', 'curvature_shift',
                       'norm_curvature', 'norm_curvature_linear', 'norm_curvature_shift',
                       'work_total', 'work_elastic', 'work_plastic', 'work_ratio', 'h_ratio']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train[engineered_features])

wcss, sil_scores = [], []
k_range = range(2, 7)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_train_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_train_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), wcss, marker='o')
axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (inertia)"); axes[0].set_title("Elbow Method")
axes[1].plot(list(k_range), sil_scores, marker='o', color='darkorange')
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette Score"); axes[1].set_title("Silhouette Score")
plt.tight_layout()
plt.show()

for k, w, s in zip(k_range, wcss, sil_scores):
    print(f"K={k}: WCSS={w:.0f}, silhouette={s:.3f}")


**An honest result, worth discussing rather than glossing over:** silhouette score actually peaks at K=2, not K=3. Let's look at what K=2 vs. K=3 each actually separate before deciding.

In [ ]:
for k in [2, 3]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_train_scaled)
    train[f'cluster_k{k}'] = km.labels_
    print(f"--- K={k}: cluster composition by true label ---")
    print(pd.crosstab(train[f'cluster_k{k}'], train['Y_name']))
    print()


**What this shows:** K=2 mostly just separates “MC carbide” (very hard, statistically distinct) from everything else — a clean, high-silhouette split, but one that throws away the matrix-vs-M6C distinction entirely. K=3 captures a more metallurgically meaningful three-way structure, even though its silhouette score is lower.

**This is exactly the Session 1 lesson about troubleshooting:** a purely statistical metric (silhouette) doesn't automatically know what's metallurgically meaningful. Domain knowledge – three real phase constituents – justifies choosing **K=3** here, matching the original paper's own choice, even though K=2 “wins” on silhouette alone.

---
## Quick Check 2

Why might relying on silhouette score alone to choose K be misleading here?

**(i)  Silhouette score is never useful for real data**
**(ii)  The statistically “cleanest” split (K=2) discards a real metallurgical distinction (M6C vs. matrix) that we know exists from domain knowledge**
**(iii)  K=2 and K=3 give identical clustering results, so it doesn't matter**

*Think about it, then check the answer below.*

**Answer: (ii)** — silhouette score only measures geometric separation in feature space; it has no way to know that matrix and M6C carbide are metallurgically distinct phases worth distinguishing. This is a real limitation of automated metrics that domain expertise needs to override.

## Step 5 — A Reusable Evaluation Pipeline

Since K-means cluster IDs are arbitrary (cluster “0” has no inherent meaning), we map each cluster to whichever true label is most common within it — then evaluate against the real validation labels.

In [ ]:
def cluster_and_evaluate(feature_list, name):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(train[feature_list])
    Xval = scaler.transform(val[feature_list])

    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Xtr)
    train_pred = km.predict(Xtr)
    val_pred = km.predict(Xval)

    # Map each cluster ID to its majority true label
    mapping = {}
    for cluster_id in np.unique(train_pred):
        mapping[cluster_id] = train['Y'][train_pred == cluster_id].mode()[0]
    val_pred_mapped = pd.Series(val_pred).map(mapping)

    f1_w = f1_score(val['Y'], val_pred_mapped, average='weighted')
    f1_m = f1_score(val['Y'], val_pred_mapped, average='macro')
    cm = confusion_matrix(val['Y'], val_pred_mapped, labels=[0, 1, 2, 3, 4])

    print(f"--- {name} ---")
    print(f"Validation F1 (weighted): {f1_w:.3f}")
    print(f"Validation F1 (macro):    {f1_m:.3f}")

    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(cm, cmap='Blues')
    labels = [inv_label_map[i] for i in range(5)]
    ax.set_xticks(range(5)); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(5)); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name}: Confusion Matrix")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.show()

    return f1_w, f1_m


## Step 6 — Feature Set A: The Traditional Baseline

In [ ]:
feature_set_A = ['Er(GPa)', 'H(GPa)']
f1w_A, f1m_A = cluster_and_evaluate(feature_set_A, "Feature Set A (baseline: Er, H)")


Notice: two of the five true phase categories (`m6c-matrix`, `m6c`) never appear in the *predicted* column at all — with only 3 clusters, the rarest classes can never be correctly captured. This is exactly why **macro F1** (which weighs every class equally) stays much lower than **weighted F1** (which is dominated by the common `matrix` class).

## Step 7 — Feature Set B: The Engineered Features

In [ ]:
f1w_B, f1m_B = cluster_and_evaluate(engineered_features, "Feature Set B (13 engineered features)")


---
## Quick Check 3

Feature Set B uses 13 engineered features derived from load-displacement curves, instead of just Er and H. Based on the F1 scores you just saw, what does this suggest?

**(i)  More features always hurt clustering, regardless of what they measure**
**(ii)  Features genuinely capturing more of the curve's shape (not just peak modulus/hardness) give the clustering more to work with, improving separation of the phases**
**(iii)  The results are identical to Feature Set A**

*Think about it, then check the answer below.*

**Answer: (ii)** — the engineered features capture shape information (elastic/plastic work split, curvature, unloading slope) that Er and H alone discard. This is the paper's own headline finding: feature engineering measurably improves clustering quality here, using the exact same algorithm.

## Step 8 — Feature Set C: Baseline + Engineered, Combined

In [ ]:
feature_set_C = feature_set_A + engineered_features
f1w_C, f1m_C = cluster_and_evaluate(feature_set_C, "Feature Set C (combined, 15 features)")


## Step 9 — Comparing All Three

In [ ]:
results = pd.DataFrame({
    'Feature Set': ['A (baseline)', 'B (engineered)', 'C (combined)'],
    'F1 weighted': [f1w_A, f1w_B, f1w_C],
    'F1 macro': [f1m_A, f1m_B, f1m_C],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
x = np.arange(3)
width = 0.35
ax.bar(x - width/2, results['F1 weighted'], width, label='F1 weighted', color='navy')
ax.bar(x + width/2, results['F1 macro'], width, label='F1 macro', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(results['Feature Set'])
ax.set_ylabel("F1 Score")
ax.set_title("Feature Engineering's Impact on Clustering Quality")
ax.legend()
plt.tight_layout()
plt.show()


---
## Quick Check 4

Feature Set C (combined, 15 features) performs almost identically to Feature Set B (13 engineered features alone) – adding Er and H back in barely changes anything. What does this suggest?

**(i)  Er and H were never useful at all**
**(ii)  The engineered features already capture most of the same information that Er and H provide, so adding them back contributes little new signal**
**(iii)  The clustering algorithm ignores extra features automatically**

*Think about it, then check the answer below.*

**Answer: (ii)** — Er and H are themselves derived from the same underlying load-displacement curve, so much of their information is already implicitly present in the engineered features. This is a common pattern in feature engineering: once you've captured the richer signal, the simpler summary statistics add little on top.

## Wrap-Up

- Applied K-means to a **real, human-labelled dataset** of 2,621 nanoindentation measurements
- Chose K=3 using elbow + silhouette **and** domain knowledge – an honest example of when statistical metrics alone (which favored K=2) need to be weighed against what we actually know about the material
- Reproduced the source paper's own core finding: engineered features (B) clearly beat the traditional Er/H baseline (A); combining both (C) adds little further
- Saw directly why **macro F1 stays low** even when **weighted F1 looks good** – with only 3 clusters, the rarest true phases can never be correctly captured

**Next session:** dimensionality reduction (PCA) on these same 13 engineered features, then re-clustering in the reduced space – and a look at what unsupervised clustering finds when applied spatially across a real indentation map.